# Conditional Statements

**Topic:** `2.1` &nbsp;|&nbsp; **Module 2: Control Flow and Loops**

*How a condition is evaluated, why truthiness is the source of most conditional bugs, and how to structure a safety-critical decision so it can be tested and reasoned about.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M2)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [02_control_flow/README.md](../README.md)

## 1. Why This Topic Matters

Every control decision a robot makes is a conditional: whether to drive, whether to arm, whether a reading is trustworthy. A condition that is quietly wrong produces no error - it simply takes the wrong branch, which is far more dangerous than a crash.

## 2. Learning Objectives

By the end of this topic you will be able to:

1. Explain that an if header accepts any expression and applies truthiness.
2. List the falsy values and predict the outcome for each.
3. Use elif to express mutually exclusive bands with a defined priority.
4. Distinguish a missing reading from a reading of zero.
5. Structure a multi-check decision as a severity-ordered guard chain.
6. Encode a growing policy as a decision table rather than control flow.

## 3. Prerequisites

This topic assumes you already have:

- Topic 1.5 Core Data Types and Type Conversion
- Topic 1.6 Operators and Expressions

## 4. Mental Model

**Mental model: a guard, not a switch.** An `if` does not enable or disable a capability; it either performs one action or skips it. That is why the safe pattern is to write the failing check first and return immediately: the first problem found is the one that stops the robot, so it is the one the operator needs to read.

| You write | The interpreter does | Common surprise |
| --- | --- | --- |
| `if value:` | apply truthiness | 0.0 and '0' are not what you assumed |
| `elif x:` | test only if every earlier test failed | order encodes priority |
| `x is None` | compare identity with the None singleton | `== None` is a style smell |
| `a if c else b` | produce one of two values | it is an expression, not a statement |

## 5. Terminology

Vocabulary used consistently throughout the course.

| Term | Meaning |
| --- | --- |
| Conditional statement | A header and suite that runs when a condition is truthy. |
| Condition | Any expression whose truthiness selects a branch. |
| Truthiness | The boolean value Python assigns to any object. |
| Falsy | One of the few values that evaluate as False. |
| Branch | One of the alternative paths a conditional offers. |
| elif | An additional branch tested only if the earlier ones failed. |
| Conditional expression | `a if condition else b`, usable as a value. |
| Guard clause | An early return that rejects invalid input early. |
| Decision table | Data that encodes a policy instead of control flow. |
| Off-by-one | A boundary error, e.g. `<` where `<=` was meant. |

## 6. Conceptual Explanation

A conditional is an expression that produces a value, and a conditional *statement* applies a rule using that value. Python's `if` takes any value, not only a bool: it applies its truthiness. That is why `if value:` and `if value is not None:` are not interchangeable, and why a legitimate reading of zero can vanish without any error being raised.

In [ ]:
battery_pct = 0.0

if battery_pct:
    print('has charge')
else:
    print('exactly empty - still a valid reading')

if battery_pct is not None:
    print('the sensor reported a value')

print('bool(0.0) =', bool(battery_pct), '- falsy')

Only a small set of values are falsy: `False`, `None`, zero of any numeric type, the empty string, and empty containers. Everything else is truthy, including `0.0`, `'0'` and `[0]`. That last pair is the trap - a non-empty list containing zero is truthy, and a string containing the character 0 is truthy, so a 'zero' check on text silently passes.

In [ ]:
values = [0, 0.0, '', '0', [], [0], {}, {'a': 0}, None, False, True]
for value in values:
    print(f'{value!r:<10} -> {"truthy" if value else "falsy"}')

> **Prefer the explicit question**
>
> Write the condition you mean. `if count > 0` says what you are testing; `if count` asks a different question and answers it differently for `0`, `'0'` and `[]`.

## 7. Formal Theory

An `if` statement is a header ending in a colon followed by a suite. The header is any expression; the interpreter applies truthiness to its value. `elif` branches are not independent tests - they are only reached when every earlier condition was false, which is why their order encodes a priority.

```text
if E1 -> suite1 ; elif E2 -> suite2 ; else -> suite_n   (first match wins)
```

The conditional expression `a if condition else b` is an expression, so it can appear inside a call, an f-string or an assignment. It is the idiomatic way to choose between two values, and it should be used only when both branches are equally simple - a chain of nested conditionals expressions is harder to read than an `if` statement.

In [ ]:
LOW, MEDIUM = 20, 50


def verdict(pct: float) -> str:
    return 'LOW' if pct < LOW else 'MEDIUM' if pct < MEDIUM else 'OK'


for pct in (10, 35, 90):
    print(f'{pct:>3}% -> {verdict(pct)}')

# a chain of comparisons in one expression
print('in range:', 0 <= 10 <= 100)
print('out of range:', 0 <= 101 <= 100)

| Value | Truthiness | Typical robotics meaning |
| --- | --- | --- |
| `0` | falsy | a count of zero events |
| `0.0` | falsy | a battery reading of exactly zero volts |
| `''` | falsy | an empty operator comment |
| `[]` | falsy | no queued commands |
| `'0'` | **truthy** | text that looks numeric but is not |
| `None` | falsy | the sensor has not reported |

## 8. Syntax

The canonical form tests the most specific case first and ends with the general fallback, so each boundary appears exactly once.

In [ ]:
LOW_BATTERY_PCT = 20
CRITICAL_BATTERY_PCT = 8


def verdict(pct: float) -> str:
    if pct <= CRITICAL_BATTERY_PCT:
        return 'CRITICAL'
    elif pct < LOW_BATTERY_PCT:
        return 'LOW'
    else:
        return 'OK'


for pct in (5, 15, 80):
    print(f'{pct:>3}% -> {verdict(pct)}')

The anti-pattern - a chain of independently evaluated `if`s:

```text
if pct < 20:
    level = 'LOW'
if pct < 50:            # also true when pct is 10
    level = 'MEDIUM'      # silently overwrites the first answer

if pct < 20:            # elif makes the priority explicit
    level = 'LOW'
elif pct < 50:
    level = 'MEDIUM'
```

> **elif is not else-if**
>
> `elif` is only evaluated when every earlier condition was false. A chain of separate `if` statements re-tests the value each time and the last match wins, which is rarely what the author intended.

## 9. Basic Examples

**Example 1 - comparisons and membership.**

In [ ]:
battery_pct = 18.0
state = {'status': 'ACTIVE', 'mode': 'auto'}

print('less    :', battery_pct < 20)
print('chained :', 0 <= battery_pct <= 100)
print('equal   :', battery_pct == 18.0)
print('not     :', not (battery_pct > 20))
print('membership:', state['status'] in {'ACTIVE', 'IDLE'})
print('identity:', state.get('mode') is None)

**Example 2 - combining conditions without nesting.**

In [ ]:
def can_drive(battery_pct: float, estop: bool) -> bool:
    return battery_pct > 20 and not estop


for pct, stop in ((50, False), (50, True), (5, False)):
    print(f'pct={pct:<3} estop={stop!s:<6} -> {can_drive(pct, stop)}')

# 'or' returns an operand, so it can supply a default
print('default  :', '' or 'rover-01')

## 10. Intermediate Examples

A geofence is the canonical robotics conditional: four bounds, one decision, and a rule about what to do when the robot is outside them.

In [ ]:
def in_geofence(x: float, y: float, bounds: tuple) -> bool:
    """Return True when (x, y) is inside the axis-aligned rectangle."""
    x_min, x_max, y_min, y_max = bounds
    return x_min <= x <= x_max and y_min <= y <= y_max


FIELD = (0.0, 50.0, 0.0, 30.0)
for x, y in ((10, 10), (60, 10), (10, 35)):
    print(f'({x:>3}, {y:>3}) inside: {in_geofence(x, y, FIELD)}')

## 11. Advanced Examples

A decision table scales better than a growing `elif` chain, because adding a case means adding data rather than editing control flow. This is the shape a safety policy should take.

In [ ]:
RULES = (
    (lambda pct, stop: stop, 'ESTOP active'),
    (lambda pct, stop: pct <= 8, 'battery critical'),
    (lambda pct, stop: pct < 20, 'battery low'),
)


def first_problem(pct: float, stop: bool):
    for test, message in RULES:
        if test(pct, stop):
            return message
    return None


for pct, stop in ((50, True), (5, False), (15, False), (90, False)):
    print(f'pct={pct:<3} stop={stop!s:<5} -> {first_problem(pct, stop)}')

The `match` statement added in Python 3.10 is another way to express multi-way branching, and it is worth knowing about - but a table of predicates scales the same way and is easier to unit test.

## 12. Code Walkthrough

Below is a safety interlock for an agricultural field robot: the decision that says whether the drive circuit may be energised at all.

In [ ]:
"""interlock.py - may the drive circuit be energised?"""
from __future__ import annotations

CRITICAL_BATTERY_PCT = 8.0
LOW_BATTERY_PCT = 20.0
MAX_TILT_DEG = 15.0


def interlock(state: dict) -> tuple[bool, str]:
    """Return (permitted, reason). The first failing check wins.

    Checks are ordered from most to least severe so the reported
    reason is the one an operator should act on first.
    """
    if state.get('estop'):
        return False, 'emergency stop is engaged'
    battery = state.get('battery_pct')
    if battery is None:
        return False, 'battery reading is missing'
    if battery <= CRITICAL_BATTERY_PCT:
        return False, 'battery is critical'
    if battery < LOW_BATTERY_PCT:
        return False, 'battery is low'
    tilt = state.get('tilt_deg', 0.0)
    if tilt > MAX_TILT_DEG:
        return False, 'robot is tilted beyond the limit'
    if not state.get('link_ok', True):
        return False, 'radio link is down'
    return True, 'all checks passed'

The ordering is the safety property: an engaged emergency stop short-circuits every later check, so the reason reported is always the most urgent one. Returning a tuple rather than printing means a test can assert the exact reason, and a supervisor can log it without parsing text.

## 13. Line-by-Line Explanation

1. The thresholds are module-level constants so the safety policy lives in one place and can be reviewed or tuned without reading logic.
2. The estop check is first and returns immediately, so no later check can override the most severe condition.
3. Using .get with an explicit None test distinguishes 'no reading' from 'reading of zero', which a truthiness check could not.
4. The critical check uses <= while the low check uses <, so the boundary value is reported as critical rather than low - the boundary is defined once.
5. The tilt check uses state.get with a zero default, because a missing tilt reading means level rather than missing.
6. Every branch returns the same shape, a (bool, str) tuple, so the caller never has to handle a special case.
7. The final return is the success case, written last so adding a new check requires no edit to it.

## 14. Common Mistakes

**Mistake 1 - a chain of independent `if` statements.**

```text
if pct < 20: level = 'LOW'
if pct < 50: level = 'MEDIUM'   # also true when pct is 10

if pct < 20: level = 'LOW'
elif pct < 50: level = 'MEDIUM' # mutually exclusive
```

**Mistake 2 - truthiness where an explicit test was meant.**

```text
if battery_pct:            # 0.0 is skipped
    publish(battery_pct)

if battery_pct is not None:
    publish(battery_pct)
```

**Mistake 3 - chaining with `and` instead of `elif` when the bands overlap.**

```text
if 0 <= pct < 20: level = 'LOW'
if 20 <= pct < 50: level = 'MEDIUM'

if pct < 50: level = 'LOW'      # catches 30 as LOW
elif pct < 20: level = 'MEDIUM' # unreachable for 30
```

**Mistake 4 - using `==` for floats read from a sensor.**

```text
if battery_pct == 20.0:   # exact equality on a measured value

if abs(battery_pct - 20.0) < 0.01:   # compare with a tolerance
```

## 15. Debugging Techniques

When a branch behaves unexpectedly, print what the condition actually evaluated to. Nine times in ten the answer is that the value is not the type or the value the author assumed.

In [ ]:
samples = [0.0, 0, '', '0', None, [], [0]]
for value in samples:
    print(f'{value!r:<8} bool={bool(value)!s:<6} is None={value is None}')

For a chain of branches, a truth table is faster than adding prints. Enumerate the cases that matter and state what each should return.

In [ ]:
CRITICAL, LOW = 8.0, 20.0

def expected(pct):
    if pct <= CRITICAL:
        return 'CRITICAL'
    if pct < LOW:
        return 'LOW'
    return 'OK'


for pct in (7.9, 8.0, 19.9, 20.0, 50.0):
    print(f'{pct:>5} -> {expected(pct)}')

> **Boundaries are where bugs live**
>
> A value exactly on a threshold is the single most common source of off-by-one defects. Test 7.9, 8.0 and 8.1 before you trust a threshold.

## 16. Best Practices

- Use `elif` for mutually exclusive bands so only one branch can win.
- Order branches from most to least severe so the first match is the most urgent.
- Prefer an explicit comparison over truthiness whenever zero is legitimate.
- Use `is None` to test for absence; use `==` to test for a value.
- Compare measured floats with a tolerance, never with `==`.
- Return a value from a decision function instead of printing inside it.
- Give every branch the same return shape so callers never special-case.
- Keep thresholds in named constants, not in the middle of the logic.

## 17. Performance Considerations

A branch is cheap. Python evaluates the condition, compares truthiness, and either enters the suite or does not - nanoseconds, with no allocation. Five thousand iterations of a five-branch chain are not a performance problem; the work inside the branches is what costs.

Where branch structure does matter is in a long chain. Because each `elif` is only tested when the earlier ones failed, ordering by likelihood is a real - if small - optimisation for a hot path. It matters far more for readability: a chain ordered by severity documents the policy, and a decision table makes the same policy data rather than control flow.

> **Measure the branch, not the body**
>
> If a conditional is suspected of being slow, the work inside it is the reason. Time the body, not the `if`.

## 18. Pythonic Approaches

Nested blocks versus a single boolean expression.

In [ ]:
battery_pct, tilt, link_ok = 18.0, 4.0, True

# before: nested blocks for one combined decision
if battery_pct > 20:
    if tilt < 15:
        if link_ok:
            permitted = True
        else:
            permitted = False
    else:
        permitted = False
else:
    permitted = False

# after: one expression, one meaning
permitted = battery_pct > 20 and tilt < 15 and link_ok
print('permitted:', permitted)

## 19. Robotics Connection

The field robot's drive circuit is gated by exactly the kind of chain this topic teaches: a set of conditions, each with a different severity, combined into one permission. The simulator supplies the readings so the policy can be tested against real values rather than invented ones.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / 'shared').is_dir()), None)
if root is not None:
    sys.path.insert(0, str(root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402

robot = SimulatedRobot(name='field-01', battery_wh=48.0)
state = robot.status()

battery = state['battery_pct']
permitted = battery > 20
print(f'battery {battery:.1f}% -> permitted: {permitted}')

empty_robot = SimulatedRobot(name='field-02', battery_wh=1.0)
print('depleted robot permitted:', empty_robot.status()['battery_pct'] > 20)

## 20. Engineering Example

A decision helper that reports which check failed, so a log line can explain the outcome rather than just recording it.

In [ ]:
def check_all(*pairs) -> tuple[bool, list[str]]:
    """Evaluate (passed, label) pairs and collect the failed labels."""
    failures = [label for passed, label in pairs if not passed]
    return (not failures), failures

## 21. Guided Practice

1. Predict the truthiness of 0, 0.0, '', '0', [] and [0] before running them.
2. Write `band(pct)` returning 'CRITICAL', 'LOW' or 'OK', then test the four boundary values around each threshold.
3. Write a two-condition guard with `and`, then the same logic with nested `if` blocks, and compare the two versions.
4. Write `classify(value)` returning 'missing', 'empty', 'number' or 'other', handling None and empty containers correctly.

In [ ]:
CRITICAL, LOW = 8.0, 20.0


def band(pct):
    if pct <= CRITICAL:
        return 'CRITICAL'
    if pct < LOW:
        return 'LOW'
    return 'OK'


for pct in (7.9, 8.0, 19.9, 20.0, 50.0):
    print(f'{pct:>5} -> {band(pct)}')

def classify(value):
    if value is None:
        return 'missing'
    if not value:
        return 'empty'
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return 'number'
    return 'other'

## 22. Knowledge Checks

Answer these before moving on. Full answers are in [solution.ipynb](solution.ipynb).

- **Q1.** What does `if value:` actually test?
- **Q1.** What does `elif` do that a second `if` statement does not?
- **Q1.** Why compare a measured float against a threshold with a tolerance?
- **Q1.** Why does the interlock check for an engaged estop before anything else?

## 23. Summary

A conditional in Python tests truthiness, not truth. Only `False`, `None`, zero, the empty string and empty containers are falsy, and that small set is where most conditional bugs live - a battery reading of `0.0` is a perfectly valid value that a bare `if battery_pct:` will silently discard.

The `elif` chain encodes a priority, not a set of independent tests. Writing separate `if` statements instead means the last match wins, which is almost never the intent, and it becomes a real defect the moment two conditions overlap. Ordering branches from most to least severe is what makes the first reported problem the one an operator should act on.

The professional pattern for anything safety-relevant is the guard chain: test the most severe condition, return immediately with a reason, and return the same shape from every branch. That structure is testable - a test asserts the exact reason, not just a boolean - and it scales: adding a new case means adding a check, not editing a growing tangle of nesting. When the branches stop fitting comfortably in an `elif` chain, move the policy into a table of predicates so it becomes data you can unit test and review.

## 24. Key Takeaways

- Python tests truthiness, so 0.0, '' and [] are all falsy.
- Use `elif` for mutually exclusive bands; the first match wins.
- Order branches by severity so the first failure is the most urgent one.
- Use `is None` to test absence, not `== None` or truthiness.
- Compare measured floats with a tolerance, never with `==`.
- Return a value from a decision function instead of printing inside it.
- Give every branch the same return shape so callers never special-case.
- Move a growing policy into a decision table once the chain gets long.

## 25. Further Exploration

- [Compound statements - the if statement](https://docs.python.org/3/reference/compound_stmts.html#the-if-statement)
- [Conditional expressions](https://docs.python.org/3/reference/expressions.html#conditional-expressions)
- [Truth value testing](https://docs.python.org/3/library/stdtypes.html#truth-value-testing)
- [PEP 8 - use a maximum line length of 79](https://peps.python.org/pep-0008/)
- [Structural pattern matching](https://docs.python.org/3/reference/compound_stmts.html#the-match-statement)